In [8]:
import datetime as dt

import matplotlib.patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
from scipy.io import readsav

In [9]:
#Load raw magnetosheath data from Thomsen et al. 2018
sheath_fp = '../data/raw/Thomsen_2018_supplementary_CassiniDataSet.txt'

# Import sheath data and clear nans
sheath_data = pd.read_csv(sheath_fp,sep="\t")
sheath_data = sheath_data.dropna().reset_index(drop=True)
print(len(sheath_data))

#Convert the sheath data to inferred upstream solar wind velocity
indexnum=100
k=1.38e-23 #Boltzmann constant in J/K (SI units)
Tp1=sheath_data['H+_temperature(eV)'][indexnum]
mp=1.67e-27 #Proton mass in kg
V1=1000.*(math.sqrt((sheath_data['H+_Vr(km/s)'][indexnum]**2)+(sheath_data['H+_Vtheta(km/s)'][indexnum]**2)+(sheath_data['H+_Vphi(km/s)'][indexnum]**2)))#in km/s
v_sw1 = (math.sqrt((2.*(1.860*((k*Tp1) + (0.5*mp*V1**2.))))/mp))/1000. #in km/s
print(v_sw1)

#Also found this file so probe what's in there
Inferred_Vsw_file='../data/raw/Thomsen_inferred_vsw_Mach.sav'
Inferred_Vsw_vals=readsav(Inferred_Vsw_file)
print(len(Inferred_Vsw_vals))
print(Inferred_Vsw_vals)


#Now create the inferred Vsw for the entire list of sheath measurements, by looping through the list
Tp=[]
V=[]
v_sw=[]

for i in range(np.shape(sheath_data)[0]):
        Tp.iloc[i] = sheath_data['H+_temperature(eV)'][i]
        

#for _,i in sheath_data.iterrows():
#    Tp.append=sheath_data['H+_temperature(eV)'][i]
   # V.append=1000.*(math.sqrt((sheath_data['H+_Vr(km/s)'][i]**2)+(sheath_data['H+_Vtheta(km/s)'][i]**2)+(sheath_data['H+_Vphi(km/s)'][i]**2)))#in km/s
   # v_sw.append = (math.sqrt((2.*(1.860*((k*Tp[i]) + (0.5*mp*V[i]**2.))))/mp))/1000. #in km/s

#print(v_sw)

19155
168.21266711622525
2
{'vsw': array([347.46873193, 351.65895407, 353.9065714 , ..., 374.3112363 ,
       374.89935927, 436.42081705], dtype='>f8'), 'machalf': array([1.55546396, 1.20973675, 1.27612661, ..., 0.63689492, 0.58964423,
       0.52491962], dtype='>f8')}


AttributeError: 'list' object has no attribute 'iloc'

In [ ]:
#Then plot distribution of v_sw inferred from all sheath measurements
fix,ax=plt.subplots()
nbins=50
#bins=np.logspace(np.log10(0.0001),np.log10(1), nbins) probably not log bins for this
ax.hist(v_sw,color='grey',density=True,alpha=0.7,histtype="step")
ax.set_xlabel("$V_SW$ (km/s) Inferred from Magnetosheath Measurements")
ax.set_ylabel("# of events")
ax.set_yscale('log')
#ax.set_ylim([1e0,5e2]) #for when we don't set density=True
#ax.set_ylim([1e-2,1e4]) #for when we set density=True to give a normalised histogram
#ax.set_xlim([0,1000])
ax.set_title('Inferred $V_SW$ distribution')

median_Vsw_sheath = np.median(np.array(v_sw)) 
mean_Vsw_sheath = np.mean(np.array(v_sw))  
print(len(v_sw))
print('median and mean of inferred Vsw for sheath measurements is: ')
print(median_Vsw_sheath, mean_Vsw_sheath)
ax.axvline(x=mean_Vsw_sheath, linewidth=2, linestyle="dashed", label=f"{len(v_sw):.0f} BS crossings: Mean: {mean_Vsw_sheath:.3f}, Median: {median_Vsw_sheath:.3f} nPa",color='grey')


In [ ]:
#Then match all the sheath measurements to the closest LFE
#(may need to be careful here as 657 distinct sheath intervals, but 19155 measurements, so we don't want to bias the distribution by searching for an LFE to match each point?)
#could match all in the first instance, and then consider removing duplicate exact values?

#Below is copied code from the matching of the BS crossings
#Note this was a dataframe (?) with lots of information including the crossing time and the inferred DP
#May need to tweak how we store the sw velocity, perhaps appending it into the sheath_data dataframe so that it can be easily matched to a time?

#overplot distribution of DP *near* LFEs that are observed between 07-11 hr LT
# For each BS/MP crossing with inferred DP, find the closest following LFE
#First search near the LFEs that are seen between LT of 07-11 hrs

lfe_match_window = dt.timedelta(hours=50)  #this can be adjusted to search over any window size
matched_events = []

for _, LFErow in LFEs_LT0711.iterrows():

    crossing_times = pd.to_datetime(BS_crossings_allDP["crossing_time"])
    lfe_start = pd.to_datetime(LFErow["start"])

    # negative is crossing is before LFE, positive if crossing is after LFE
    time_differences = crossing_times - lfe_start
    
    #Yesprint(time_differences)

    # We are only interested in the crossings prior to the LFE
    time_differences = time_differences[time_differences < dt.timedelta(0)]

    if len(time_differences) ==0:
        continue
    
    # The closest crossing to the LFE is the one with the smallest time difference
    closest_crossing_index = time_differences.idxmax()
    closest_crossing_time_difference = time_differences[closest_crossing_index]

    # The event is a match if the time difference is less than our matching window
    if abs(closest_crossing_time_difference) <= lfe_match_window:
        matched_events.append(
            {
                "LFE": LFErow,  # This LFE
                "Crossing": BS_crossings_allDP.loc[
                    closest_crossing_index
                ],  # The matching crossing
            }
        )

    else:
        continue

print(f"{len(matched_events)} matched LFEs for 07-11 all")
dp=[]
for event in matched_events:
    dp.append(event["Crossing"]["inf_DP"])